In [2]:
import requests
import pandas as pd
import os
import time
import shutil

# ========================
# MONTAR GOOGLE DRIVE
# ========================
from google.colab import drive
drive.mount('/content/drive')

# Caminho base no Drive
BASE_DIR = "/content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/"
os.makedirs(BASE_DIR, exist_ok=True)

# ========================
# CONFIGURAÇÕES
# ========================
BASE_URL = "https://www.sodimac.com.br/s/search/v1/sobr/category/products"
CATEGORY_ID = "cat620002"  # pisos e revestimentos

OUTPUT_FILE = os.path.join(BASE_DIR, "sodimac_produtos.csv")
IMAGES_DIR = os.path.join(BASE_DIR, "sodimac_imagens")
ZIP_FILE = os.path.join(BASE_DIR, "sodimac_imagens.zip")

LIMIT = 10  # limite de produtos (troque para float("inf") para todos)

HEADERS = {"User-Agent": "Mozilla/5.0"}

# ========================
# FUNÇÃO PARA PEGAR PÁGINA
# ========================
def get_products(page=1):
    params = {
        "priceGroup": "1018",
        "zone": "35745",
        "currentpage": page,
        "channel": "kiosk",
        "region": "35",
        "city": "9575",
        "categoryId": CATEGORY_ID
    }
    try:
        r = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=10)
        r.raise_for_status()
        return r.json().get("data", {}).get("results", [])
    except Exception as e:
        print(f"❌ Erro na página {page}: {e}")
        return []

# ========================
# COLETA PRODUTOS
# ========================
dados = []
page = 1

while len(dados) < LIMIT:
    print(f"🔄 Coletando página {page}...")
    produtos = get_products(page)

    if not produtos:
        break

    for p in produtos:
        if len(dados) >= LIMIT:
            break

        codigo = p.get("productId", "")
        nome = p.get("displayName", "")

        # Preço (EVENT > NORMAL > M2)
        preco = None
        for pr in p.get("prices", []):
            if pr.get("type") in ["EVENT", "NORMAL", "M2"]:
                preco = pr.get("price")
                break

        descricao = p.get("model", "")
        especificacoes = "; ".join(
            [f"{h['key']}: {h['value']}" for h in p.get("highlights", [])]
        )

        imagens = p.get("mediaUrls", [])
        imagem = imagens[-1] if imagens else None

        url = f"https://www.sodimac.com.br/sodimac-br/product/{codigo}/"
        loja = "Sodimac"

        dados.append([codigo, nome, preco, descricao, especificacoes, imagem, url, loja])

    page += 1
    time.sleep(1)

# ========================
# SALVAR CSV NO DRIVE
# ========================
colunas = [
    "Código",
    "Nome",
    "Preço Promocional",
    "Descrição",
    "Especificações",
    "Imagem",
    "URL",
    "Loja",
]
df = pd.DataFrame(dados, columns=colunas)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"✅ Coleta finalizada! {len(df)} produtos salvos em {OUTPUT_FILE}")

# ========================
# BAIXAR IMAGENS
# ========================
os.makedirs(IMAGES_DIR, exist_ok=True)

for i, row in df.iterrows():
    codigo = row["Código"]
    img_url = row["Imagem"]

    if not img_url:
        print(f"⚠️ Produto {codigo} sem imagem.")
        continue

    try:
        r = requests.get(img_url, stream=True, timeout=10)
        if r.status_code == 200:
            img_path = os.path.join(IMAGES_DIR, f"{codigo}.jpg")
            with open(img_path, "wb") as f:
                shutil.copyfileobj(r.raw, f)
            print(f"📸 Imagem salva: {img_path}")
        else:
            print(f"❌ Erro ao baixar imagem {codigo}: {r.status_code}")
    except Exception as e:
        print(f"❌ Erro ao baixar imagem {codigo}: {e}")

# ========================
# COMPACTAR IMAGENS
# ========================
shutil.make_archive(IMAGES_DIR, "zip", IMAGES_DIR)
print(f"✅ Imagens salvas e compactadas em {ZIP_FILE}")


Mounted at /content/drive
🔄 Coletando página 1...
✅ Coleta finalizada! 10 produtos salvos em /content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/sodimac_produtos.csv
📸 Imagem salva: /content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/sodimac_imagens/762339.jpg
📸 Imagem salva: /content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/sodimac_imagens/866091.jpg
📸 Imagem salva: /content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/sodimac_imagens/753136.jpg
📸 Imagem salva: /content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/sodimac_imagens/700353.jpg
📸 Imagem salva: /content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/sodimac_imagens/869955.jpg
📸 Imagem salva: /content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/sodimac_imagens/784034.jpg
📸 Imagem salva: /content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_SODIMAC/sodimac_imagens/762355.jpg
📸 Imagem salva: /content/drive/MyDrive/WebScraping/FEITO/SODIMAC/COLETA_S